### Connection

In [ ]:
import os
import pandas as pd 
import numpy as np 
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="pandas only supports SQLAlchemy connectable"
)

!pip install psycopg2-binary
import psycopg2
 
# Redshift cluster details
host = os.environ["REDSHIFT_HOST"]
port = 5439
dbname = os.environ["REDSHIFT_DB"]
user = os.environ["REDSHIFT_USER"]
password = os.environ["REDSHIFT_PASSWORD"]
 
# Establish a connection to Redshift
try:
    conn = psycopg2.connect(
        dbname=dbname,
        user=user,
        password=password,
        host=host,
        port=port
    )
    print("Connection to Redshift established successfully.")
except Exception as e:
    print(f"Error connecting to Redshift: {e}")

### Engine

In [ ]:
query1 = """
SELECT * FROM RAW_SOURCE_DATA.INCIDENTS
"""
query2 = """
SELECT * FROM RAW_SOURCE_DATA.UP_CALENDAR
"""
query3 = """
SELECT * FROM RAW_SOURCE_DATA.WEATHER_DATA
"""
#query4 = """
#SELECT * FROM RAW_SOURCE_DATA.ANOMALIES
#"""

incidents_df = pd.read_sql(query1,conn)
holidays_df = pd.read_sql(query2, conn)
weather_df = pd.read_sql(query3, conn)
#anomalies_df = pd.read_sql(query4, conn)
#incidents_df
#holidays_df
#weather_df

query5 = """
SELECT * FROM RAW_SOURCE_DATA.cases
"""

anomalies_df = pd.read_csv('anomalous_trips.csv')
anomalies_df

import pandas as pd
from datetime import datetime

# STEP 1: Ensure datetime parsing for anomalies and holidays
anomalies_df['start_time'] = pd.to_datetime(anomalies_df['start_time'])  # If your column is 'start_time', change accordingly
holidays_df['date'] = pd.to_datetime(holidays_df['date'])

# STEP 2: Convert all event/alert date fields to string for safe parsing
incidents_df['event_date'] = incidents_df['event_date'].astype(str)
weather_df['alert_date'] = weather_df['alert_date'].astype(str)

# STEP 3: Custom parser for handling single date or range (e.g. "2023-11-18 to 2023-11-26")
def parse_event_dates(date_val):
    date_str = str(date_val)  # Convert to string in case it's not already

    if 'to' in date_str:
        try:
            start_str, end_str = date_str.split('to')
            start_date = datetime.strptime(start_str.strip(), "%Y-%m-%d").date()
            end_date = datetime.strptime(end_str.strip(), "%Y-%m-%d").date()
            return pd.Series([start_date, end_date])
        except Exception as e:
            print(f"Error parsing range: {date_str} -> {e}")
            return pd.Series([None, None])
    else:
        try:
            date_obj = datetime.strptime(date_str.strip(), "%Y-%m-%d").date()
            return pd.Series([date_obj, date_obj])
        except Exception as e:
            print(f"Error parsing single date: {date_str} -> {e}")
            return pd.Series([None, None])

# STEP 4: Apply parser to incidents and weather data
incidents_df[['start_date', 'end_date']] = incidents_df['event_date'].apply(parse_event_dates)
weather_df[['start_date', 'end_date']] = weather_df['alert_date'].apply(parse_event_dates)

# STEP 5: Rule engine logic
def check_rules(row):
    trip_date = row['start_time'].date()
    remarks = []

    # 1. Weekend check
    if row['start_time'].weekday() >= 5:
        remarks.append("Weekend")

    # 2. Holiday check
    holiday_match = holidays_df[holidays_df['date'].dt.date == trip_date]
    if not holiday_match.empty and holiday_match.iloc[0]['is_holiday']:
        remarks.append(f"Holiday: {holiday_match.iloc[0]['holiday_name']}")

    # 3. Incident check
    incident_match = incidents_df[
        (incidents_df['start_date'] <= trip_date) &
        (incidents_df['end_date'] >= trip_date)
    ]
    if not incident_match.empty:
        for event in incident_match['event_name']:
            remarks.append(f"Incident: {event}")

    # 4. Weather alert check
    weather_match = weather_df[
        (weather_df['start_date'] <= trip_date) &
        (weather_df['end_date'] >= trip_date)
    ]
    if not weather_match.empty:
        for alert in weather_match['alert']:
            remarks.append(f"Weather Alert: {alert}")

    if remarks:
        return pd.Series([True, "; ".join(remarks)])
    else:
        return pd.Series([False, "No special conditions"])

# STEP 6: Apply rule engine to each anomaly
anomalies_df[['is_explained', 'remarks']] = anomalies_df.apply(check_rules, axis=1)

# STEP 7: Filter only unexplained anomalies
final_anomalies_df = anomalies_df[~anomalies_df['is_explained']].copy()
final_anomalies_df.drop(columns=['is_explained'], inplace=True)

# STEP 8: Display final result
from IPython.display import display

print("Final unexplained anomalies:")
display(final_anomalies_df)

final_anomalies_df.to_csv("final_anomalies_df.csv", index=False)

In [ ]:

anomalies_df = pd.read_csv('anomalous_trips.csv')
anomalies_df




import pandas as pd
from datetime import datetime

# STEP 1: Ensure datetime parsing for anomalies and holidays
anomalies_df['start_time'] = pd.to_datetime(anomalies_df['start_time'])  # If your column is 'start_time', change accordingly
holidays_df['date'] = pd.to_datetime(holidays_df['date'])

# STEP 2: Convert all event/alert date fields to string for safe parsing
incidents_df['event_date'] = incidents_df['event_date'].astype(str)
weather_df['alert_date'] = weather_df['alert_date'].astype(str)

# STEP 3: Custom parser for handling single date or range (e.g. "2023-11-18 to 2023-11-26")
def parse_event_dates(date_val):
    date_str = str(date_val)  # Convert to string in case it's not already

    if 'to' in date_str:
        try:
            start_str, end_str = date_str.split('to')
            start_date = datetime.strptime(start_str.strip(), "%Y-%m-%d").date()
            end_date = datetime.strptime(end_str.strip(), "%Y-%m-%d").date()
            return pd.Series([start_date, end_date])
        except Exception as e:
            print(f"Error parsing range: {date_str} -> {e}")
            return pd.Series([None, None])
    else:
        try:
            date_obj = datetime.strptime(date_str.strip(), "%Y-%m-%d").date()
            return pd.Series([date_obj, date_obj])
        except Exception as e:
            print(f"Error parsing single date: {date_str} -> {e}")
            return pd.Series([None, None])

# STEP 4: Apply parser to incidents and weather data
incidents_df[['start_date', 'end_date']] = incidents_df['event_date'].apply(parse_event_dates)
weather_df[['start_date', 'end_date']] = weather_df['alert_date'].apply(parse_event_dates)

# STEP 5: Rule engine logic
def check_rules(row):
    trip_date = row['start_time'].date()
    remarks = []

    # 1. Weekend check
    if row['start_time'].weekday() >= 5:
        remarks.append("Weekend")

    # 2. Holiday check
    holiday_match = holidays_df[holidays_df['date'].dt.date == trip_date]
    if not holiday_match.empty and holiday_match.iloc[0]['is_holiday']:
        remarks.append(f"Holiday: {holiday_match.iloc[0]['holiday_name']}")

    # 3. Incident check
    incident_match = incidents_df[
        (incidents_df['start_date'] <= trip_date) &
        (incidents_df['end_date'] >= trip_date)
    ]
    if not incident_match.empty:
        for event in incident_match['event_name']:
            remarks.append(f"Incident: {event}")

    # 4. Weather alert check
    weather_match = weather_df[
        (weather_df['start_date'] <= trip_date) &
        (weather_df['end_date'] >= trip_date)
    ]
    if not weather_match.empty:
        for alert in weather_match['alert']:
            remarks.append(f"Weather Alert: {alert}")

    if remarks:
        return pd.Series([True, "; ".join(remarks)])
    else:
        return pd.Series([False, "No special conditions"])

# STEP 6: Apply rule engine to each anomaly
anomalies_df[['is_explained', 'remarks']] = anomalies_df.apply(check_rules, axis=1)

# STEP 7: Filter only unexplained anomalies
final_anomalies_df = anomalies_df[~anomalies_df['is_explained']].copy()
final_anomalies_df.drop(columns=['is_explained'], inplace=True)

# STEP 8: Display final result
from IPython.display import display

print("Final unexplained anomalies:")
display(final_anomalies_df)

final_anomalies_df.to_csv("final_anomalies_df.csv", index=False)

In [ ]:
transformed_trip_ids = final_anomalies_df['trip_id']

In [ ]:
final_anomalies_df

In [ ]:
actual_ids = []
for trips in transformed_trip_ids:
    original_trip_ids = [round((tid + 97) / 42) for tid in transformed_trip_ids]
    actual_ids.append(original_trip_ids)

In [ ]:
print(len(actual_ids))

In [ ]:
# List your transformed trip IDs here:
transformed_trip_ids = [363473825]  # Replace with your actual transformed IDs

# Reverse the transformation:
original_trip_ids = [round((tid + 97) / 42) for tid in transformed_trip_ids]

# Print results:
print(original_trip_ids)
print("total trip ids:", len(original_trip_ids))

In [ ]:
new_df = pd.read_csv("s3://transit-gov2-da-dev/for_model/Transit Data/data_1.csv")
new_df

In [ ]:
final_anomalies_df

In [ ]:
new_df['anomaly'] = new_df['trip_id'].isin(final_anomalies_df['trip_id']).astype(int)

print(new_df)

In [ ]:
new_df.to_csv("new_df.csv", index = False)

In [ ]:
new_df.to_csv("s3://transit-gov2-da-dev/for_model/Transit Data/new_df.csv", index = False)

In [ ]:
new_df['anomaly'] = new_df['trip_id'].isin(final_anomalies_df['trip_id']).astype(int)

print(new_df)

anomalies_count = (new_df['anomaly'] == 1).sum()
anomalies_count

In [ ]:
final_anomalies_df

In [ ]:
final_anomalies_df.shape

In [ ]:
new_df.shape

In [ ]:
anomalous_trip_majority_voting = pd.read_csv("anomalous_trips_majority_voting.csv")

In [ ]:
anomalous_trip_majority_voting.shape

In [ ]:
count_anomalies = (new_df['anomaly'] == 1).sum()
print(count_anomalies)

In [ ]:
final_df['anomaly'] = final_df['trip_id'].isin(anomalies).astype(int)


In [ ]:
data1_df = pd.read_csv("s3://transit-gov2-da-dev/for_model/Transit Data/data_1.csv")

In [ ]:
data1_df.shape

In [ ]:
data1_df.head(5)

In [ ]:
final_anomalies_df.shape

In [ ]:
anomalous_trip_ids = final_anomalies_df['trip_id']
print(len(anomalous_trip_ids))

In [ ]:
data_df = data_df[data_df['route_id'].isin([54, 55])]

In [ ]:
data_df.shape

In [ ]:
new_df['anomaly'] = new_df['trip_id'].isin(final_anomalies_df['trip_id']).astype(int)

print(new_df)

anomalies_count = (new_df['anomaly'] == 1).sum()
anomalies_count

In [ ]:
data_df = data_df
data_df.shape

In [ ]:
data_df2['axnomaly'] = data_df2['trip_id'].isin(final_anomalies_df['trip_id']).astype(int)

In [ ]:
anomalies_count = (data_df2['anomaly'] == 1).sum()
anomalies_count

In [ ]:
len(anomalous_trip_ids)

In [ ]:
data_df = data_df2

In [ ]:
import numpy as np

data_df2['anomaly'] = np.where(
    data_df2['trip_id'].isin(anomalous_trip_ids), 1, 0
)

In [ ]:
count_anomalies = data_df2['anomaly'].sum()
print(count_anomalies)

In [ ]:
final_df = pd.read_csv("anomalous_trips_majority_voting.csv")

In [ ]:
final_df.shape

In [ ]:
final_anomaly_trip = final_df['trip_id']

In [ ]:
len(final_anomaly_trip)

In [ ]:
count = data_df2[
    (data_df2['anomaly'] == 1) &
    (data_df2['trip_id'].isin(final_anomaly_trip))
].shape[0]

print(count)